# Cat and Dog Image Classifier

Completed solution for the freeCodeCamp **Machine Learning with Python** project.

Run the notebook from top to bottom in Google Colab. The final cell evaluates the 50 test images and requires at least **63% accuracy**.


In [ ]:
try:
    # This command only works in Google Colab.
    get_ipython().run_line_magic("tensorflow_version", "2.x")
except Exception:
    pass

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


In [ ]:
# Get project files.
# Remove a previous copy first so the notebook can be re-run safely in Colab.
!rm -rf cats_and_dogs cats_and_dogs.zip
!wget -q https://cdn.freecodecamp.org/project-data/cats-and-dogs/cats_and_dogs.zip
!unzip -q cats_and_dogs.zip

PATH = "cats_and_dogs"

train_dir = os.path.join(PATH, "train")
validation_dir = os.path.join(PATH, "validation")
test_dir = os.path.join(PATH, "test")

total_train = sum(len(files) for _, _, files in os.walk(train_dir))
total_val = sum(len(files) for _, _, files in os.walk(validation_dir))
total_test = len(os.listdir(test_dir))

batch_size = 64
epochs = 20
IMG_HEIGHT = 150
IMG_WIDTH = 150

print("Training images:", total_train)
print("Validation images:", total_val)
print("Test images:", total_test)


In [ ]:
# 3
# Rescale all pixels from [0, 255] to [0, 1].
train_image_generator = ImageDataGenerator(rescale=1.0 / 255)
validation_image_generator = ImageDataGenerator(rescale=1.0 / 255)
test_image_generator = ImageDataGenerator(rescale=1.0 / 255)

train_data_gen = train_image_generator.flow_from_directory(
    directory=train_dir,
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    seed=SEED,
)

val_data_gen = validation_image_generator.flow_from_directory(
    directory=validation_dir,
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    seed=SEED,
)

# The test images live directly under cats_and_dogs/test rather than under
# cats_and_dogs/test/cats and cats_and_dogs/test/dogs. Therefore PATH is the
# directory passed to flow_from_directory and classes=["test"] identifies the
# single test subdirectory. shuffle=False preserves freeCodeCamp's expected order.
test_data_gen = test_image_generator.flow_from_directory(
    directory=PATH,
    classes=["test"],
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None,
    shuffle=False,
)


In [ ]:
# 4
def plotImages(images_arr, probabilities=False):
    count = len(images_arr)
    fig, axes = plt.subplots(count, 1, figsize=(5, count * 3))

    # plt.subplots returns a single Axes object when count == 1.
    if count == 1:
        axes = [axes]

    if probabilities is False:
        for img, ax in zip(images_arr, axes):
            ax.imshow(img)
            ax.axis("off")
    else:
        for img, probability, ax in zip(images_arr, probabilities, axes):
            ax.imshow(img)
            ax.axis("off")
            if probability > 0.5:
                ax.set_title(f"{probability * 100:.2f}% dog")
            else:
                ax.set_title(f"{(1 - probability) * 100:.2f}% cat")

    plt.tight_layout()
    plt.show()


sample_training_images, _ = next(train_data_gen)
plotImages(sample_training_images[:5])


In [ ]:
# 5
# Data augmentation reduces overfitting by creating random variations of the
# relatively small training set.
train_image_generator = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    fill_mode="nearest",
)


In [ ]:
# 6
train_data_gen = train_image_generator.flow_from_directory(
    batch_size=batch_size,
    directory=train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    seed=SEED,
)

augmented_images = [train_data_gen[0][0][0] for _ in range(5)]
plotImages(augmented_images)


In [ ]:
# 7
model = Sequential([
    Conv2D(
        32,
        (3, 3),
        padding="same",
        activation="relu",
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    ),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), padding="same", activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), padding="same", activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
# 8
# Re-create validation iterator so it starts from the beginning.
val_data_gen.reset()

history = model.fit(
    x=train_data_gen,
    steps_per_epoch=max(1, total_train // batch_size),
    epochs=epochs,
    validation_data=val_data_gen,
    validation_steps=max(1, total_val // batch_size),
    verbose=1,
)


In [ ]:
# 9
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(len(acc))

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")

plt.tight_layout()
plt.show()


In [ ]:
# 10
# Predict in the exact filename order expected by the freeCodeCamp test.
test_data_gen.reset()

probabilities = (
    model.predict(
        test_data_gen,
        steps=int(np.ceil(total_test / batch_size)),
        verbose=1,
    )
    .reshape(-1)
    .tolist()
)

# Get the same ordered images for visualization.
test_data_gen.reset()
test_images = next(test_data_gen)

print("Predictions:", len(probabilities))
plotImages(test_images[:total_test], probabilities[:total_test])


In [ ]:
# 11
answers = [
    1, 0, 0, 1, 0, 0, 0, 0, 1, 1,
    0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
    0, 0, 1, 1, 1, 1, 1, 0, 0, 0,
    0, 0, 1, 1, 0, 1, 1, 1, 1, 0,
    1, 0, 1, 1, 0, 0, 0, 0, 0, 0,
]

correct = 0

for probability, answer in zip(probabilities, answers):
    if round(probability) == answer:
        correct += 1

percentage_identified = (correct / len(answers)) * 100
passed_challenge = percentage_identified >= 63

print(
    f"Your model correctly identified "
    f"{round(percentage_identified, 2)}% of the images of cats and dogs."
)

if passed_challenge:
    print("You passed the challenge!")
else:
    print(
        "You haven't passed yet. Your model should identify at least 63% "
        "of the images. Keep trying. You will get it!"
    )
